# M1.0d Primer: Coordinate Systems and the Integral Theorems

Book pointer: chapter 1, section 1.5 (coordinate systems), and the divergence and Stokes theorems from sections 1.3 and 1.4.

Last of four primer notebooks. Two topics. First, cylindrical and spherical coordinates: why they exist, how their unit vectors behave, and where the extra factors in the gradient, divergence and curl formulas come from. Second, the two theorems that connect the local operators of notebook 00c to the integrals of notebook 00b. They are the bridge between the differential and integral forms of every law in the course.

## 1. Warm-up

1. Convert the point $(x, y, z) = (1, 1, 2)$ to cylindrical and to spherical coordinates.
2. Why is $\hat{\mathbf{r}}$ in cylindrical coordinates not a constant vector? Write it in terms of $\hat{\mathbf{x}}$ and $\hat{\mathbf{y}}$.
3. The distance moved when $\phi$ changes by $d\phi$ is not $d\phi$. What is it, in cylindrical and in spherical coordinates?
4. State the divergence theorem and Stokes' theorem. What kind of region and boundary does each involve?
5. Why does a field with zero curl have a potential?

## 2. The idea in one paragraph

Cartesian coordinates describe every point with three distances along fixed directions. Cylindrical and spherical coordinates instead use distances and angles measured from an axis or a centre, which turns problems with that symmetry into problems with fewer variables. The price is that the unit vectors turn as you move, and the derivatives must account for the fact that a change $d\phi$ in an angle moves you a distance $r\,d\phi$, not $d\phi$. The integral theorems are a separate idea: they say that if you add up the divergence (or curl) over every tiny box (or loop) inside a region, the contributions from internal faces (or edges) cancel in pairs, leaving only the boundary. That is what lets Gauss' law and Ampere's law be written either as statements about derivatives at a point or as statements about integrals over surfaces and loops.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=5, suppress=True)

### 3.1 Cylindrical coordinates $(r, \phi, z)$

Replace $x$ and $y$ by the distance $r$ from the $z$ axis and the angle $\phi$ measured from the $x$ axis; keep $z$. Conversions:
$$x = r\cos\phi,\quad y = r\sin\phi, \qquad r = \sqrt{x^2 + y^2},\quad \phi = \operatorname{atan2}(y, x).$$
(Use the two-argument arctangent so that $\phi$ lands in the right quadrant.)

The unit vectors point in the direction of increasing coordinate: $\hat{\mathbf{r}}$ straight away from the axis, $\hat{\boldsymbol{\phi}}$ counter-clockwise around it, $\hat{\mathbf{z}}$ up.
$$\hat{\mathbf{r}} = \cos\phi\,\hat{\mathbf{x}} + \sin\phi\,\hat{\mathbf{y}}, \qquad \hat{\boldsymbol{\phi}} = -\sin\phi\,\hat{\mathbf{x}} + \cos\phi\,\hat{\mathbf{y}}.$$
They are perpendicular and of unit length at every point, but they turn as $\phi$ changes. This has a consequence that trips everyone once: a field that is "constant" in cylindrical components, say $\mathbf{F} = \hat{\boldsymbol{\phi}}$, is not a constant vector field. It is the rotation field scaled to unit length.

The line element: moving by $dr$ covers distance $dr$; moving by $d\phi$ covers $r\,d\phi$ (an arc of radius $r$); moving by $dz$ covers $dz$. So
$$d\mathbf{l} = dr\,\hat{\mathbf{r}} + r\,d\phi\,\hat{\boldsymbol{\phi}} + dz\,\hat{\mathbf{z}}.$$
The factors $1, r, 1$ are called the scale factors. They are the whole reason the differential operators look different in these coordinates.

**Worked example 1.** $(x, y, z) = (1, 1, 2)$: $r = \sqrt2 = 1.414$, $\phi = 45^\circ = \pi/4$, $z = 2$. At that point $\hat{\mathbf{r}} = (0.707, 0.707, 0)$ and $\hat{\boldsymbol{\phi}} = (-0.707, 0.707, 0)$. The radial field $(x, y, 0)$ there is $(1, 1, 0) = \sqrt2\,\hat{\mathbf{r}} = r\hat{\mathbf{r}}$. The rotation field $(-y, x, 0) = (-1, 1, 0) = \sqrt2\,\hat{\boldsymbol{\phi}} = r\hat{\boldsymbol{\phi}}$.

**Worked example 2.** Convert the Cartesian field $\mathbf{F} = (-y, x, 0)/(x^2 + y^2)$ (the wire field from notebook 00c) to cylindrical components: $F_r = \mathbf{F}\cdot\hat{\mathbf{r}} = (-y\cos\phi + x\sin\phi)/r^2 = (-r\sin\phi\cos\phi + r\cos\phi\sin\phi)/r^2 = 0$, and $F_\phi = \mathbf{F}\cdot\hat{\boldsymbol{\phi}} = (y\sin\phi + x\cos\phi)/r^2 = r/r^2 = 1/r$. So $\mathbf{F} = \hat{\boldsymbol{\phi}}/r$: three symbols instead of a fraction with four.

In [ ]:
def to_cyl(p):
    x, y, z = p[..., 0], p[..., 1], p[..., 2]
    return np.hypot(x, y), np.arctan2(y, x), z

def cyl_unit_vectors(phi):
    r_hat = np.stack([np.cos(phi), np.sin(phi), np.zeros_like(phi)], axis=-1)
    phi_hat = np.stack([-np.sin(phi), np.cos(phi), np.zeros_like(phi)], axis=-1)
    return r_hat, phi_hat

p = np.array([1.0, 1.0, 2.0])
r, phi, z = to_cyl(p)
r_hat, phi_hat = cyl_unit_vectors(phi)
print(f"(1, 1, 2) -> r = {r:.4f}, phi = {np.degrees(phi):.1f} deg, z = {z}")
print("r_hat =", r_hat, " phi_hat =", phi_hat, " perpendicular:", r_hat @ phi_hat, " unit:", np.linalg.norm(r_hat))
print("radial field (x, y, 0) =", np.array([p[0], p[1], 0]), " = r * r_hat =", r * r_hat)
print("rotation field (-y, x, 0) =", np.array([-p[1], p[0], 0]), " = r * phi_hat =", r * phi_hat)

# Worked example 2: cylindrical components of the wire field at several points.
pts = np.array([[1.0, 1.0, 0], [2.0, 0.0, 0], [-1.0, 3.0, 0], [0.5, -0.5, 0]])
F = np.stack([-pts[:, 1], pts[:, 0], np.zeros(4)], axis=-1) / (pts[:, 0]**2 + pts[:, 1]**2)[:, None]
rr, pp, _ = to_cyl(pts); rh, ph = cyl_unit_vectors(pp)
print("wire field: F_r =", np.einsum("ij,ij->i", F, rh), "  F_phi =", np.einsum("ij,ij->i", F, ph), "  1/r =", 1 / rr)

# Picture: unit vectors at several points.
fig, ax = plt.subplots(figsize=(6, 6))
ang = np.linspace(0, 2 * np.pi, 13)[:-1]; rad = np.array([0.7, 1.4] * 6)
px, py = rad * np.cos(ang), rad * np.sin(ang)
rh, ph = cyl_unit_vectors(ang)
ax.quiver(px, py, rh[:, 0], rh[:, 1], color="C3", scale=8, label="r-hat"); ax.quiver(px, py, ph[:, 0], ph[:, 1], color="C0", scale=8, label="phi-hat")
for rr_ in (0.7, 1.4): ax.add_patch(plt.Circle((0, 0), rr_, fill=False, ls=":", color="grey"))
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.legend(); ax.set_title("cylindrical unit vectors turn with phi"); plt.show()

### 3.2 Spherical coordinates $(r, \theta, \phi)$

Now $r$ is the distance from the origin, $\theta$ the polar angle down from the $z$ axis ($0$ at the north pole, $\pi$ at the south pole), and $\phi$ the azimuth around the $z$ axis as before.
$$x = r\sin\theta\cos\phi,\quad y = r\sin\theta\sin\phi,\quad z = r\cos\theta; \qquad r = \sqrt{x^2+y^2+z^2},\quad \theta = \arccos(z/r),\quad \phi = \operatorname{atan2}(y, x).$$

Unit vectors: $\hat{\mathbf{r}}$ away from the origin, $\hat{\boldsymbol{\theta}}$ southward along a meridian, $\hat{\boldsymbol{\phi}}$ eastward along a circle of latitude.
$$\hat{\mathbf{r}} = (\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta),\quad \hat{\boldsymbol{\theta}} = (\cos\theta\cos\phi,\ \cos\theta\sin\phi,\ -\sin\theta),\quad \hat{\boldsymbol{\phi}} = (-\sin\phi,\ \cos\phi,\ 0).$$

Scale factors: $dr$ moves you $dr$; $d\theta$ moves you along a meridian of radius $r$, so $r\,d\theta$; $d\phi$ moves you along a circle of latitude whose radius is $r\sin\theta$, so $r\sin\theta\,d\phi$.
$$d\mathbf{l} = dr\,\hat{\mathbf{r}} + r\,d\theta\,\hat{\boldsymbol{\theta}} + r\sin\theta\,d\phi\,\hat{\boldsymbol{\phi}}.$$
Multiplying the two angular lengths gives the surface element on a sphere, $dS = r^2\sin\theta\,d\theta\,d\phi$ (derived from the cross product in notebook 00b), and multiplying all three gives the volume element $dv = r^2\sin\theta\,dr\,d\theta\,d\phi$.

**Worked example 3.** $(1, 1, 2)$: $r = \sqrt6 = 2.449$, $\theta = \arccos(2/2.449) = 35.3^\circ$, $\phi = 45^\circ$. A step of $d\phi = 0.01$ at this point moves a distance $r\sin\theta\,d\phi = 2.449\cdot0.577\cdot0.01 = 0.01414$, which equals $\sqrt{x^2 + y^2}\,d\phi = 1.414\cdot0.01$: the radius of the latitude circle is the cylindrical $r$, as it should be.

In [ ]:
def to_sph(p):
    x, y, z = p[..., 0], p[..., 1], p[..., 2]
    r = np.sqrt(x**2 + y**2 + z**2)
    return r, np.arccos(z / r), np.arctan2(y, x)

def sph_unit_vectors(theta, phi):
    r_hat = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
    th_hat = np.array([np.cos(theta) * np.cos(phi), np.cos(theta) * np.sin(phi), -np.sin(theta)])
    ph_hat = np.array([-np.sin(phi), np.cos(phi), 0.0])
    return r_hat, th_hat, ph_hat

p = np.array([1.0, 1.0, 2.0]); r, th, ph = to_sph(p)
print(f"(1, 1, 2) -> r = {r:.4f}, theta = {np.degrees(th):.1f} deg, phi = {np.degrees(ph):.1f} deg")
rh, thh, phh = sph_unit_vectors(th, ph)
print("r_hat . theta_hat =", round(rh @ thh, 12), " r_hat . phi_hat =", round(rh @ phh, 12), " theta_hat . phi_hat =", round(thh @ phh, 12))
print("r * r_hat recovers the point:", r * rh)

# Scale factor check: the distance moved for a small change in each coordinate.
def from_sph(r, th, ph): return np.array([r * np.sin(th) * np.cos(ph), r * np.sin(th) * np.sin(ph), r * np.cos(th)])
d = 0.01
print(f"step dr = {d}: moved {np.linalg.norm(from_sph(r + d, th, ph) - p):.5f}  (scale factor 1)")
print(f"step dtheta = {d}: moved {np.linalg.norm(from_sph(r, th + d, ph) - p):.5f}  (r dtheta = {r * d:.5f})")
print(f"step dphi = {d}: moved {np.linalg.norm(from_sph(r, th, ph + d) - p):.5f}  (r sin(theta) dphi = {r * np.sin(th) * d:.5f})")

### 3.3 Gradient, divergence and curl in curvilinear coordinates

The operators keep their meanings (slope, flux per volume, circulation per area) and only their formulas change, and the scale factors explain every change.

Gradient. The directional derivative along $\hat{\boldsymbol{\phi}}$ is the change in $f$ per unit distance, and a change $d\phi$ covers distance $r\,d\phi$ (cylindrical), so the $\hat{\boldsymbol{\phi}}$ component of $\nabla f$ is $\frac1r\frac{\partial f}{\partial\phi}$:
$$\nabla f = \frac{\partial f}{\partial r}\hat{\mathbf{r}} + \frac1r\frac{\partial f}{\partial\phi}\hat{\boldsymbol{\phi}} + \frac{\partial f}{\partial z}\hat{\mathbf{z}} \quad\text{(cylindrical)}, \qquad \nabla f = \frac{\partial f}{\partial r}\hat{\mathbf{r}} + \frac1r\frac{\partial f}{\partial\theta}\hat{\boldsymbol{\theta}} + \frac{1}{r\sin\theta}\frac{\partial f}{\partial\phi}\hat{\boldsymbol{\phi}} \quad\text{(spherical)}.$$

Divergence. Flux per volume, but now the box faces have different areas on the near and far side. In cylindrical coordinates the faces at $r$ and $r + dr$ have areas $r\,d\phi\,dz$ and $(r + dr)\,d\phi\,dz$, so the net radial flux is $d(rF_r)\,d\phi\,dz$ and after dividing by the volume $r\,dr\,d\phi\,dz$ the radial term is $\frac1r\frac{\partial(rF_r)}{\partial r}$. The same reasoning in spherical coordinates, where the radial faces have area $r^2\sin\theta\,d\theta\,d\phi$, gives $\frac1{r^2}\frac{\partial(r^2F_r)}{\partial r}$:
$$\nabla\cdot\mathbf{F} = \frac1r\frac{\partial(rF_r)}{\partial r} + \frac1r\frac{\partial F_\phi}{\partial\phi} + \frac{\partial F_z}{\partial z} \quad\text{(cyl.)}, \qquad \nabla\cdot\mathbf{F} = \frac1{r^2}\frac{\partial(r^2F_r)}{\partial r} + \frac{1}{r\sin\theta}\frac{\partial(\sin\theta F_\theta)}{\partial\theta} + \frac{1}{r\sin\theta}\frac{\partial F_\phi}{\partial\phi} \quad\text{(sph.)}.$$

Curl, $z$ component in cylindrical coordinates (the one used most): $\frac1r\left(\frac{\partial(rF_\phi)}{\partial r} - \frac{\partial F_r}{\partial\phi}\right)$. The $r$ inside the derivative comes from the loop's outer edge being longer than its inner edge. The full set is in `study/vector_calculus_ref.md`.

**Worked example 4.** Divergence of the radial field $\mathbf{F} = r\hat{\mathbf{r}}$ (cylindrical, $F_r = r$): $\frac1r\frac{d(r\cdot r)}{dr} = \frac1r\cdot2r = 2$. In Cartesian: $\partial_x x + \partial_y y = 2$. Same.

**Worked example 5.** Curl of the rotation field $\mathbf{F} = r\hat{\boldsymbol{\phi}}$ ($F_\phi = r$): $\frac1r\frac{d(r\cdot r)}{dr} = 2$. Of the wire field $\hat{\boldsymbol{\phi}}/r$ ($F_\phi = 1/r$): $\frac1r\frac{d(r\cdot1/r)}{dr} = \frac1r\frac{d(1)}{dr} = 0$ for $r > 0$. That is the zero curl measured in notebook 00c, now in one line.

**Worked example 6.** Divergence of the Coulomb field $\hat{\mathbf{r}}/r^2$ (spherical, $F_r = r^{-2}$): $\frac1{r^2}\frac{d(r^2\cdot r^{-2})}{dr} = \frac1{r^2}\frac{d(1)}{dr} = 0$ for $r > 0$. The field of a point charge has no divergence anywhere except at the charge. Compare the divergence of $\mathbf{F} = r^2\hat{\mathbf{r}}$: $\frac1{r^2}\frac{d(r^4)}{dr} = 4r$; in Cartesian, $\mathbf{F} = r(x, y, z)$ and $\nabla\cdot\mathbf{F} = 3r + (x\cdot x + y\cdot y + z\cdot z)/r = 3r + r = 4r$.

The code checks worked examples 4 to 6 by computing each divergence or curl in Cartesian components on a grid and comparing with the one-line curvilinear result.

In [ ]:
s = np.linspace(0.5, 2.5, 41); hs = s[1] - s[0]
Xg, Yg = np.meshgrid(s, s, indexing="ij")
Rc = np.hypot(Xg, Yg)
def div2(Fx, Fy): return np.gradient(Fx, hs, axis=0) + np.gradient(Fy, hs, axis=1)
def curlz(Fx, Fy): return np.gradient(Fy, hs, axis=0) - np.gradient(Fx, hs, axis=1)
inner = (slice(2, -2), slice(2, -2))
print(f"div of r r_hat  : grid {div2(Xg, Yg)[inner].mean():.4f}            cylindrical formula 2")
print(f"curl of r phi_hat: grid {curlz(-Yg, Xg)[inner].mean():.4f}           cylindrical formula 2")
print(f"curl of phi_hat/r: grid max |.| {np.abs(curlz(-Yg / Rc**2, Xg / Rc**2)[inner]).max():.1e}   cylindrical formula 0")

X3, Y3, Z3 = np.meshgrid(s, s, s, indexing="ij"); R3 = np.sqrt(X3**2 + Y3**2 + Z3**2)
def div3(F): return sum(np.gradient(F[k], hs, axis=k) for k in range(3))
inner3 = (slice(2, -2),) * 3
print(f"div of r_hat/r^2 : grid max |.| {np.abs(div3([X3 / R3**3, Y3 / R3**3, Z3 / R3**3])[inner3]).max():.1e}   spherical formula 0")
d4 = div3([R3 * X3, R3 * Y3, R3 * Z3])
print(f"div of r^2 r_hat : grid / (4r) = {(d4 / (4 * R3))[inner3].mean():.4f} (spread {(d4 / (4 * R3))[inner3].std():.1e})   spherical formula 4r")

### 3.4 The divergence theorem

$$\int_v \nabla\cdot\mathbf{F}\,dv = \oint_S \mathbf{F}\cdot d\mathbf{S}.$$

The integral of the divergence over a volume equals the outward flux through its closed boundary surface. Why: fill the volume with tiny boxes. By the definition of divergence in notebook 00c, $\nabla\cdot\mathbf{F}\,dv$ is the flux out of each tiny box. Add them all up. Every internal face is shared by two boxes, and what leaves one box through that face enters the other, so the two contributions cancel. Only the faces on the outer boundary survive, and their sum is the flux through $S$. The picture below shows the cancellation for four boxes.

Physical reading: the total amount of source inside a region equals the net amount flowing out through its boundary. In chapter 4, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ integrated over a volume becomes $\oint\mathbf{E}\cdot d\mathbf{S} = Q_{\text{enc}}/\epsilon_0$: Gauss' law.

**Worked example 7.** $\mathbf{F} = (x^2, yz, z)$ over the unit cube $[0, 1]^3$. $\nabla\cdot\mathbf{F} = 2x + z + 1$. Volume integral: $\int_0^1\!\int_0^1\!\int_0^1(2x + z + 1)\,dx\,dy\,dz = 1 + \tfrac12 + 1 = 2.5$. Flux through the six faces: $x = 1$ face, normal $+\hat{\mathbf{x}}$: $\int F_x = \int 1 = 1$. $x = 0$ face: $-\int 0 = 0$. $y = 1$: $\int F_y\,dx\,dz = \int z\,dx\,dz = \tfrac12$. $y = 0$: $-\int 0 = 0$. $z = 1$: $\int F_z = 1$. $z = 0$: $-\int 0 = 0$. Total $2.5$.

In [ ]:
# The cancellation picture: four boxes, flux arrows on every face. Internal faces carry equal and opposite arrows.
fig, ax = plt.subplots(figsize=(6, 6))
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((i, j), 1, 1, fill=False, lw=2))
        cx, cy = i + 0.5, j + 0.5
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ax.annotate("", xy=(cx + 0.5 * dx + 0.25 * dx, cy + 0.5 * dy + 0.25 * dy), xytext=(cx + 0.5 * dx - 0.1 * dx, cy + 0.5 * dy - 0.1 * dy),
                        arrowprops=dict(arrowstyle="->", color="C3" if (0 < cx + 0.5 * dx < 2 and 0 < cy + 0.5 * dy < 2) else "C0", lw=2))
ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.6, 2.6); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("red: internal faces, each crossed twice in opposite directions and cancelling\nblue: boundary faces, the only ones that survive the sum"); plt.show()

# Worked example 7, numerically.
F7 = lambda p: np.stack([p[:, 0]**2, p[:, 1] * p[:, 2], p[:, 2]], axis=-1)
n = 200
s = (np.arange(n) + 0.5) / n
Xc, Yc, Zc = np.meshgrid(s, s, s, indexing="ij")
vol_int = np.sum(2 * Xc + Zc + 1) / n**3
U, V = np.meshgrid(s, s, indexing="ij"); dA = 1 / n**2
flux_total = 0.0
for axis in range(3):
    for val, sign in ((1.0, 1.0), (0.0, -1.0)):
        pts = np.zeros((n * n, 3)); others = [k for k in range(3) if k != axis]
        pts[:, others[0]] = U.ravel(); pts[:, others[1]] = V.ravel(); pts[:, axis] = val
        flux_total += sign * np.sum(F7(pts)[:, axis]) * dA
print(f"volume integral of div F = {vol_int:.5f}    flux through the cube's surface = {flux_total:.5f}    by hand: 2.5")

### 3.5 Stokes' theorem

$$\int_S (\nabla\times\mathbf{F})\cdot d\mathbf{S} = \oint_C \mathbf{F}\cdot d\mathbf{l}.$$

The flux of the curl through an open surface equals the circulation of the field around the surface's edge. Same mechanism as before, one dimension down: tile the surface with tiny loops. The curl times the loop area is the circulation around each tiny loop. Every internal edge is traversed by two neighbouring loops in opposite directions and cancels. What survives is the boundary curve $C$. The orientation rule: with the right-hand thumb along $d\mathbf{S}$, the fingers give the direction of travel around $C$.

Physical reading: the total "spin" inside a surface shows up as circulation around its rim. In chapter 12, $\nabla\times\mathbf{B} = \mu_0\mathbf{J}$ integrated over a surface becomes $\oint\mathbf{B}\cdot d\mathbf{l} = \mu_0 I_{\text{enc}}$: Ampere's law. In chapter 14, Faraday's law gets its integral form the same way.

**Worked example 8.** $\mathbf{F} = (-y, x, z)$, surface: the unit disk in the $z = 0$ plane with normal $+\hat{\mathbf{z}}$, boundary: the unit circle counter-clockwise. $\nabla\times\mathbf{F} = (0 - 0,\ 0 - 0,\ 1 - (-1)) = (0, 0, 2)$. Flux of the curl through the disk: $2\times\pi = 2\pi$. Circulation: on the circle $\mathbf{F} = (-\sin t, \cos t, 0)$ and $d\mathbf{l} = (-\sin t, \cos t)\,dt$, so $\oint\mathbf{F}\cdot d\mathbf{l} = \int_0^{2\pi}1\,dt = 2\pi$.

**Worked example 9.** Same field, but the surface is the upper hemisphere of the unit sphere, which has the same boundary circle. Stokes' theorem says the answer must again be $2\pi$: the flux of a curl depends only on the rim. Check: on the hemisphere $\hat{\mathbf{n}} = \hat{\mathbf{r}}$ and $(\nabla\times\mathbf{F})\cdot\hat{\mathbf{r}} = 2\cos\theta$, so the flux is $\int 2\cos\theta\,\sin\theta\,d\theta\,d\phi = 2\pi\cdot2\cdot\tfrac12 = 2\pi$.

In [ ]:
F8 = lambda p: np.stack([-p[:, 1], p[:, 0], p[:, 2]], axis=-1)
curl_F8 = lambda p: np.tile([0.0, 0.0, 2.0], (len(p), 1))

def surface_patches(r_of_uv, u_range, v_range, nu, nv):
    u = u_range[0] + (np.arange(nu) + 0.5) * (u_range[1] - u_range[0]) / nu
    v = v_range[0] + (np.arange(nv) + 0.5) * (v_range[1] - v_range[0]) / nv
    du, dv = (u_range[1] - u_range[0]) / nu, (v_range[1] - v_range[0]) / nv
    Ug, Vg = np.meshgrid(u, v, indexing="ij"); e = 1e-6
    ru = (r_of_uv(Ug + e, Vg) - r_of_uv(Ug - e, Vg)) / (2 * e); rv = (r_of_uv(Ug, Vg + e) - r_of_uv(Ug, Vg - e)) / (2 * e)
    return r_of_uv(Ug, Vg).reshape(-1, 3), (np.cross(ru, rv) * du * dv).reshape(-1, 3)

disk = lambda s, phi: np.stack([s * np.cos(phi), s * np.sin(phi), np.zeros_like(s)], axis=-1)       # normal +z
hemi = lambda th, phi: np.stack([np.sin(th) * np.cos(phi), np.sin(th) * np.sin(phi), np.cos(th)], axis=-1)  # normal r_hat
c1, d1 = surface_patches(disk, (0, 1), (0, 2 * np.pi), 200, 400)
c2, d2 = surface_patches(hemi, (0, np.pi / 2), (0, 2 * np.pi), 200, 400)
t = np.linspace(0, 2 * np.pi, 4001); rim = np.stack([np.cos(t), np.sin(t), 0 * t], axis=-1)
circ = np.sum(F8(0.5 * (rim[1:] + rim[:-1])) * np.diff(rim, axis=0))
print(f"flux of curl F through the disk:        {np.sum(curl_F8(c1) * d1):.5f}")
print(f"flux of curl F through the hemisphere:  {np.sum(curl_F8(c2) * d2):.5f}")
print(f"circulation of F around the shared rim: {circ:.5f}      2 pi = {2 * np.pi:.5f}")

### 3.6 Two consequences

**Curl-free fields have a potential.** If $\nabla\times\mathbf{F} = 0$ everywhere, Stokes' theorem makes every closed-loop integral zero, so the line integral between two points is the same along every path. Define $V(\mathbf{r}) = -\int_{\mathbf{r}_0}^{\mathbf{r}}\mathbf{F}\cdot d\mathbf{l}$ along any path; it is a well-defined scalar field and $\mathbf{F} = -\nabla V$. This is exactly how chapter 3 constructs the electric potential from the electrostatic field. The code does it: integrate $\mathbf{F} = (y, x)$ from the origin to every grid point along a straight line, then take the gradient and recover $\mathbf{F}$.

**The wire-field puzzle, resolved.** $\mathbf{F} = \hat{\boldsymbol{\phi}}/r$ has zero curl for $r > 0$ but circulation $2\pi$ around any loop enclosing the $z$ axis. Stokes' theorem does not fail; it does not apply. The field is undefined on the axis, so no surface spanning the loop can avoid a point where the curl is undefined. The theorem holds for loops that do not enclose the axis, and the code checks that too. Physically the whole circulation comes from the axis itself: that is where the current is, and Ampere's law counts it.

In [ ]:
# Potential of a curl-free field by line integration, then recover the field from the gradient.
xg = np.linspace(-1, 1, 41); hg = xg[1] - xg[0]
Xg, Yg = np.meshgrid(xg, xg, indexing="ij")
F_cons = lambda p: np.stack([p[:, 1], p[:, 0]], axis=-1)        # (y, x), curl-free
tt = np.linspace(0, 1, 201)[:, None]
V = np.zeros_like(Xg)
for i in range(Xg.shape[0]):
    for j in range(Xg.shape[1]):
        path = tt * np.array([Xg[i, j], Yg[i, j]])
        dl = np.diff(path, axis=0); mid = 0.5 * (path[1:] + path[:-1])
        V[i, j] = -np.sum(F_cons(mid) * dl)
Fx_back, Fy_back = [-g for g in np.gradient(V, hg, hg)]
print(f"V built by line integration equals -xy:  max |V + xy| = {np.abs(V + Xg * Yg).max():.1e}")
print(f"-grad V recovers F = (y, x):  max error {max(np.abs(Fx_back - Yg)[2:-2, 2:-2].max(), np.abs(Fy_back - Xg)[2:-2, 2:-2].max()):.1e}")

# The wire field: circulation around loops that do and do not enclose the axis.
F_wire = lambda p: np.stack([-p[:, 1], p[:, 0]], axis=-1) / (p[:, 0]**2 + p[:, 1]**2)[:, None]
def loop_integral(F, centre, radius):
    t = np.linspace(0, 2 * np.pi, 4001); path = np.array(centre) + radius * np.stack([np.cos(t), np.sin(t)], axis=-1)
    return np.sum(F(0.5 * (path[1:] + path[:-1])) * np.diff(path, axis=0))
for centre, radius in (((0, 0), 1.0), ((0, 0), 0.3), ((1.5, 0.5), 0.6), ((2.0, 2.0), 1.9), ((2.0, 0.0), 2.5)):
    enclosed = np.hypot(*centre) < radius
    print(f"loop centre {centre}, radius {radius}: circulation = {loop_integral(F_wire, centre, radius):+.5f}   axis {'inside' if enclosed else 'outside'} -> expect {2 * np.pi if enclosed else 0:.5f}")

### Exercise: Gauss' law before chapter 4

Replace the body of `enclosed_source` so that it integrates the divergence of $\mathbf{F} = (x^2, yz, z)$ over the box $[a_x, b_x]\times[a_y, b_y]\times[a_z, b_z]$ with a midpoint-rule grid sum. The check compares it with the outward flux through the box's surface computed by the function above it, for a box that is not the unit cube.

In [ ]:
def box_flux(F, lo, hi, n=150):
    """Outward flux of F through the surface of the box lo <= r <= hi."""
    total = 0.0
    for axis in range(3):
        others = [k for k in range(3) if k != axis]
        u = lo[others[0]] + (np.arange(n) + 0.5) * (hi[others[0]] - lo[others[0]]) / n
        v = lo[others[1]] + (np.arange(n) + 0.5) * (hi[others[1]] - lo[others[1]]) / n
        Ug, Vg = np.meshgrid(u, v, indexing="ij"); dA = (hi[others[0]] - lo[others[0]]) * (hi[others[1]] - lo[others[1]]) / n**2
        for val, sign in ((hi[axis], 1.0), (lo[axis], -1.0)):
            pts = np.zeros((n * n, 3)); pts[:, others[0]] = Ug.ravel(); pts[:, others[1]] = Vg.ravel(); pts[:, axis] = val
            total += sign * np.sum(F(pts)[:, axis]) * dA
    return total

def enclosed_source(div_F, lo, hi, n=150):
    """Midpoint-rule integral of div F over the box."""
    axes = [lo[k] + (np.arange(n) + 0.5) * (hi[k] - lo[k]) / n for k in range(3)]       # replace from here with your own version
    Xb, Yb, Zb = np.meshgrid(*axes, indexing="ij")
    cell = np.prod([(hi[k] - lo[k]) / n for k in range(3)])
    return np.sum(div_F(Xb, Yb, Zb)) * cell


lo, hi = np.array([-0.5, 0.2, -1.0]), np.array([1.5, 1.2, 0.5])
lhs = enclosed_source(lambda x, y, z: 2 * x + z + 1, lo, hi)
rhs = box_flux(F7, lo, hi)
print(f"integral of div F over the box = {lhs:.5f}    flux out of the box = {rhs:.5f}")
assert abs(lhs - rhs) < 1e-3

## 4. Verify

The Coulomb field one last time, now with both theorems in hand. $\mathbf{F} = \hat{\mathbf{r}}/r^2$ has zero divergence for $r > 0$ (worked example 6), yet its flux through any closed surface around the origin is $4\pi$ (notebook 00b). By the divergence theorem the volume integral of $\nabla\cdot\mathbf{F}$ over the enclosed ball must therefore be $4\pi$, even though the integrand is zero everywhere except one point. The divergence is concentrated at the origin as a delta function: all of the source sits at the charge. That is the content of $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ for a point charge, and it is why the flux is the same for every surface around it.

The check: flux through concentric spheres and through a cube, all $4\pi$, and through a sphere that excludes the origin, zero.

In [ ]:
F_coulomb = lambda p: p / np.linalg.norm(p, axis=1, keepdims=True) ** 3
sphere = lambda th, phi, R, c: R * hemi(th, phi) + np.array(c)
for R, c in ((0.5, (0, 0, 0)), (1.0, (0, 0, 0)), (2.0, (0, 0, 0)), (1.0, (3.0, 0, 0))):
    cs, ds = surface_patches(lambda t, p: sphere(t, p, R, c), (0, np.pi), (0, 2 * np.pi), 300, 600)
    print(f"sphere radius {R} centred at {c}: flux = {np.sum(F_coulomb(cs) * ds):.5f}")
print(f"cube [-1, 1]^3:                  flux = {box_flux(F_coulomb, np.array([-1., -1, -1]), np.array([1., 1, 1]), n=400):.5f}    4 pi = {4 * np.pi:.5f}")

### Where each tool is used in the course

| tool | first appears | law or object |
|---|---|---|
| dot product, $\mathbf{F}\cdot d\mathbf{l}$ | ch. 3 | potential difference, emf (ch. 14) |
| cross product | ch. 11, 12 | Lorentz force, Biot-Savart |
| line integral | ch. 3, 12, 14 | potential, Ampere's law, Faraday's law |
| flux integral | ch. 4, 12, 14 | Gauss' law, magnetic flux |
| volume integral | ch. 2, 8, 15 | field of a charge distribution, field energy |
| gradient | ch. 3 | $\mathbf{E} = -\nabla V$ |
| divergence | ch. 4, 5, 12 | $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$, bound charge, $\nabla\cdot\mathbf{B} = 0$ |
| curl | ch. 12, 13, 14 | $\nabla\times\mathbf{B} = \mu_0\mathbf{J}$, bound current, $\nabla\times\mathbf{E} = -\partial_t\mathbf{B}$ |
| Laplacian | ch. 6 | Poisson and Laplace equations |
| divergence theorem | ch. 4 | integral form of Gauss' law |
| Stokes' theorem | ch. 12, 14 | integral forms of Ampere's and Faraday's laws |
| cylindrical / spherical | ch. 4, 12 | symmetric Gauss and Ampere problems |

## 5. Explore and rebuild

Explore. A uniformly charged ball of radius $R$ has field $\mathbf{E} = \hat{\mathbf{r}}\,r/3\epsilon_0\cdot\rho$ inside and $\hat{\mathbf{r}}\,\rho R^3/3\epsilon_0 r^2$ outside (set $\rho = \epsilon_0 = 1$). Compute its divergence inside and outside with the spherical formula, then compute the flux through spheres of radius $0.5R$, $R$ and $2R$ numerically. Confirm that the flux through each sphere equals the volume integral of the divergence inside it. This is Gauss' law for the ball, one chapter early.

Rebuild. Close this notebook. On paper, derive the cylindrical divergence formula's radial term from the flux through a small wedge-shaped box, and give the two-sentence argument for why the divergence theorem works (internal faces cancel). In a blank editor, write `box_flux` and the midpoint volume integral from scratch and verify worked example 7.